# TalkToTheCell — SmolVLA fine-tune on `so101-sim-pickplace`

Fine-tunes **SmolVLA (450M)** — a vision-language-action model — on
[`ahmedsohail2003/so101-sim-pickplace`](https://huggingface.co/datasets/ahmedsohail2003/so101-sim-pickplace):
100 language-labeled MuJoCo demonstrations of an SO-ARM100 arm performing
*"Pick up the red block and place it in the blue tray."*

The trained policy is pushed to `ahmedsohail2003/smolvla-so101-pickplace` automatically.

**Kaggle setup (required before running):**
1. Accelerator: **GPU T4 x2** (or P100) — Settings panel on the right
2. Internet: **ON** (Settings → Internet) — needs a phone-verified account
3. Secret: **Add-ons → Secrets → `HF_TOKEN`** = your Hugging Face *write* token

Free-tier engineering: SmolVLA's LeRobot defaults already freeze the vision
encoder and train the action expert only (`freeze_vision_encoder=True`,
`train_expert_only=True`), so the 450M model fine-tunes comfortably on a 16 GB
T4 with AMP. Expect **~4–8 h for 20k steps**; checkpoints save every 5k steps,
so a dead session can resume (see last cell).

In [ ]:
# ~3-4 min: LeRobot 0.6.0 (same version the dataset was recorded with).
# Extras: smolvla (VLA deps) + dataset (av video decoding — required by lerobot.datasets)
!pip install -q "lerobot[smolvla,dataset]==0.6.0"
!nvidia-smi

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

login(UserSecretsClient().get_secret("HF_TOKEN"))
print("logged in as:", whoami()["name"])

In [ ]:
# Sanity: the dataset loads from the Hub (metadata only, fast)
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata("ahmedsohail2003/so101-sim-pickplace")
print(meta.total_episodes, "episodes,", meta.total_frames, "frames,", meta.fps, "fps")
print("cameras:", list(meta.camera_keys))

In [ ]:
# The fine-tune. ~4-8 h on a T4 for 20k steps at batch 8.
# - policy.path pulls the pretrained SmolVLA base
# - defaults freeze the VLM; only the action expert trains
# - push_to_hub uploads the final policy + checkpoints' best to your account
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=ahmedsohail2003/so101-sim-pickplace \
  --batch_size=8 \
  --steps=20000 \
  --save_freq=5000 \
  --log_freq=100 \
  --num_workers=2 \
  --policy.use_amp=true \
  --policy.device=cuda \
  --policy.push_to_hub=true \
  --policy.repo_id=ahmedsohail2003/smolvla-so101-pickplace \
  --output_dir=/kaggle/working/train_smolvla \
  --wandb.enable=false

In [ ]:
# Confirm the model landed on the Hub
from huggingface_hub import HfApi

files = HfApi().list_repo_files("ahmedsohail2003/smolvla-so101-pickplace")
print("\n".join(files))
print("\nhttps://huggingface.co/ahmedsohail2003/smolvla-so101-pickplace")

## If the session dies mid-run

Checkpoints are in `/kaggle/working/train_smolvla/checkpoints/`. Save the run's
output as a Kaggle dataset (File → Save Version keeps `/kaggle/working`), then
in a fresh session re-run the install/login cells and resume with:

```
!lerobot-train --config_path=<checkpoint_dir>/pretrained_model/train_config.json --resume=true
```

## After training (back on the local machine)

Evaluate in the MuJoCo work-cell against the ACT baseline (65% / 75% w/ ensembling)
and record the before/after demo video — SmolVLA inference fits the local RTX 4050.